# EGO Analysis - Gemini Recommendations

This notebook implements advanced analysis methods suggested by Gemini AI.

**Reference:** `GEMINI_ANALIZ_ONERILERI.md`

## Contents:

1. **Occupancy Heatmap** - Route × Date occupancy matrix
2. **Operational Efficiency** - Ghost routes, overcrowded routes
3. **Regional Performance** - District-based analysis
4. **Route Text Mining** - Hub detection
5. **Metro Integration Analysis** - Metro-bus relationship
6. **Anomaly Detection** - Statistical anomaly identification
7. **Alert and Warning System** - Red/Green/Yellow alerts

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('Libraries loaded ✓')

In [ ]:
# Load Data
df = pd.read_csv(r"../data/ego_data_clean.csv")

df['TARIH'] = pd.to_datetime(df['TARIH'])

print(f"Data loaded: {len(df):,} rows")
print(f"Date range: {df['TARIH'].min().date()} → {df['TARIH'].max().date()}")
print(f"Unique routes: {df['HAT NO'].nunique()}")
print(f"Unique dates: {df['TARIH'].nunique()} days")
print(f"\nColumns: {list(df.columns)}")
display(df.head())

---
# 1. Occupancy Heatmap

**Gemini Recommendation:**
> "Which routes are overcrowded on which days with occupancy rate heatmap in Route × Date matrix?"

**Goal:**
- Identify critical routes
- See time-based occupancy patterns
- Detect capacity issues

In [ ]:
# 1.1 - Occupancy Heatmap Over Time for Top 20 Busiest Routes

# Top 20 busiest routes
top_20_routes = df.groupby('HAT NO')['TAŞINAN YOLCU SAYISI'].sum().nlargest(20).index

# Route × Date pivot table
df_top20 = df[df['HAT NO'].isin(top_20_routes)]
pivot_occupancy = df_top20.pivot_table(
    index='HAT NO',
    columns='TARIH',
    values='DOLULUK ORANI',
    aggfunc='mean'
)

# Visualization
fig, ax = plt.subplots(figsize=(20, 10))

sns.heatmap(pivot_occupancy, 
            cmap='RdYlGn_r',  # Red = crowded, Green = empty
            center=50,
            vmin=0, vmax=100,
            cbar_kws={'label': 'Occupancy Rate (%)'},
            linewidths=0.5,
            linecolor='gray',
            ax=ax)

ax.set_title('Daily Occupancy Heatmap for Top 20 Routes', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Route No', fontsize=12)

# X-axis date format
date_labels = [d.strftime('%d.%m') if i % 30 == 0 else '' 
               for i, d in enumerate(pivot_occupancy.columns)]
ax.set_xticklabels(date_labels, rotation=45)

plt.tight_layout()
plt.show()

print("\nHEATMAP INTERPRETATION:")
print("- Red areas: High occupancy (70%+) - May need capacity increase")
print("- Green areas: Low occupancy (30%-) - Can optimize trips")
print("- Vertical patterns: System-wide congestion on specific days")
print("- Horizontal patterns: Consistently busy/empty routes")

In [ ]:
# 1.2 - Weekly Occupancy Pattern (Heatmap)

# Average occupancy by day of week and route
df['DAY_OF_WEEK'] = df['TARIH'].dt.dayofweek
df['DAY_NAME'] = df['TARIH'].dt.day_name()

pivot_weekly = df[df['HAT NO'].isin(top_20_routes)].pivot_table(
    index='HAT NO',
    columns='DAY_OF_WEEK',
    values='DOLULUK ORANI',
    aggfunc='mean'
)

# Visualization
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(pivot_weekly,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn_r',
            center=50,
            vmin=0, vmax=100,
            cbar_kws={'label': 'Average Occupancy (%)'},
            ax=ax)

ax.set_title('Average Occupancy by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Route No')
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.show()

print("\nWEEKLY PATTERN ANALYSIS:")
print("- Weekday pattern (Mon-Fri): Commute/school traffic")
print("- Weekend pattern (Sat-Sun): Social/recreation traffic")
print("- Saturday vs Sunday difference: Shopping/leisure behavior")

---
# 2. Operational Efficiency and Alert System

**Gemini Recommendation:**
> "Ghost routes (under 5%), Overcrowded routes (over 85%), Efficiency score"

**Alert Levels:**
- 🔴 **Red:** 90%+ occupancy - Urgent capacity increase
- 🟡 **Yellow:** 70-90% occupancy - Monitoring required
- 🟢 **Green:** 10%- occupancy - Savings opportunity

In [ ]:
# 2.1 - Route-based Operational Scorecard

# Metrics for each route
route_metrics = df.groupby('HAT NO').agg({
    'TAŞINAN YOLCU SAYISI': ['sum', 'mean'],
    'DOLULUK ORANI': ['mean', 'max', 'std'],
    'SEFER SAYISI': 'sum',
    'ARAÇ KAPASİTESİ': 'sum'
})

route_metrics.columns = ['Total_Passengers', 'Avg_Daily_Passengers', 'Avg_Occupancy', 
                            'Max_Occupancy', 'Occupancy_Std', 'Total_Trips', 'Total_Capacity']

# Efficiency metrics
route_metrics['Passengers_Per_Trip'] = route_metrics['Total_Passengers'] / route_metrics['Total_Trips']
route_metrics['Capacity_Utilization'] = (route_metrics['Total_Passengers'] / 
                                         route_metrics['Total_Capacity'] * 100)

# Determine alert level
def alert_level(occupancy):
    if occupancy >= 90:
        return 'Red (Critical)'
    elif occupancy >= 70:
        return 'Yellow (Monitor)'
    elif occupancy <= 10:
        return 'Green (Savings)'
    else:
        return 'Normal'

route_metrics['Alert'] = route_metrics['Avg_Occupancy'].apply(alert_level)

# Efficiency category
route_metrics['Efficiency_Category'] = pd.cut(
    route_metrics['Passengers_Per_Trip'],
    bins=[0, 10, 30, 100],
    labels=['Low', 'Medium', 'High']
)

print("="*80)
print("OPERATIONAL SCORECARD")
print("="*80)
print(f"Total Routes: {len(route_metrics)}")
print(f"\nAlert Distribution:")
print(route_metrics['Alert'].value_counts())
print(f"\nEfficiency Distribution:")
print(route_metrics['Efficiency_Category'].value_counts())

# Most critical routes
print("\n" + "="*80)
print("🔴 CRITICAL ROUTES (Occupancy 90%+)")
print("="*80)
critical = route_metrics[route_metrics['Alert'] == 'Red (Critical)'].sort_values('Avg_Occupancy', ascending=False)
if len(critical) > 0:
    print(critical[['Avg_Occupancy', 'Max_Occupancy', 'Passengers_Per_Trip', 'Total_Trips']].head(10))
else:
    print("No critical routes.")

# Ghost routes
print("\n" + "="*80)
print("🟢 GHOST ROUTES (Occupancy 10%-)")
print("="*80)
ghost = route_metrics[route_metrics['Alert'] == 'Green (Savings)'].sort_values('Avg_Occupancy')
if len(ghost) > 0:
    print(ghost[['Avg_Occupancy', 'Passengers_Per_Trip', 'Total_Trips']].head(10))
else:
    print("No ghost routes.")

In [ ]:
# 2.2 - Alert Distribution Visualization

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Alert distribution (pie chart)
alert_counts = route_metrics['Alert'].value_counts()
colors = {'Red (Critical)': '#e74c3c', 'Yellow (Monitor)': '#f39c12', 
          'Green (Savings)': '#27ae60', 'Normal': '#95a5a6'}
pie_colors = [colors.get(x, '#95a5a6') for x in alert_counts.index]

axes[0, 0].pie(alert_counts, labels=alert_counts.index, autopct='%1.1f%%',
               colors=pie_colors, startangle=90)
axes[0, 0].set_title('Route Alert Status Distribution', fontsize=14, fontweight='bold')

# 2. Occupancy distribution histogram
axes[0, 1].hist(route_metrics['Avg_Occupancy'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 1].axvline(90, color='red', linestyle='--', linewidth=2, label='Critical (90%)')
axes[0, 1].axvline(70, color='orange', linestyle='--', linewidth=2, label='Monitor (70%)')
axes[0, 1].axvline(10, color='green', linestyle='--', linewidth=2, label='Savings (10%)')
axes[0, 1].set_title('Occupancy Rate Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Average Occupancy (%)')
axes[0, 1].set_ylabel('Number of Routes')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Efficiency scatter
scatter = axes[1, 0].scatter(route_metrics['Avg_Occupancy'], 
                             route_metrics['Passengers_Per_Trip'],
                             c=route_metrics['Total_Passengers'],
                             cmap='viridis', s=100, alpha=0.6, edgecolors='black')
axes[1, 0].set_title('Occupancy vs Passengers Per Trip', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Average Occupancy (%)')
axes[1, 0].set_ylabel('Passengers Per Trip')
axes[1, 0].axvline(90, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axvline(10, color='green', linestyle=':', alpha=0.5)
axes[1, 0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 0], label='Total Passengers')

# 4. Top 10 critical and ghost routes
top_critical = critical.head(5) if len(critical) > 0 else pd.DataFrame()
top_ghost = ghost.head(5) if len(ghost) > 0 else pd.DataFrame()

if len(top_critical) > 0:
    axes[1, 1].barh(top_critical.index.astype(str), top_critical['Avg_Occupancy'], 
                    color='#e74c3c', alpha=0.7, label='Critical')
if len(top_ghost) > 0:
    axes[1, 1].barh(top_ghost.index.astype(str), top_ghost['Avg_Occupancy'], 
                    color='#27ae60', alpha=0.7, label='Ghost')
axes[1, 1].set_title('Top 5 Critical and Ghost Routes', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Average Occupancy (%)')
axes[1, 1].set_ylabel('Route No')
axes[1, 1].legend()
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
# 3. Regional Performance Analysis

**Gemini Recommendation:**
> "Regional density map with groupings like Sincan Region Routes, Çankaya Region Routes"

**Approach:**
- Extract region names from route text
- Group routes by regions
- Calculate regional performance metrics

In [ ]:
# 3.1 - Region Labeling (Route Text Based)

# Main districts/regions of Ankara
regions = {
    'Sincan': ['sincan', 'törekent', 'yenikent'],
    'Çankaya': ['çankaya', 'kızılay', 'tunalı', 'bahçelievler', 'emek'],
    'Keçiören': ['keçiören', 'aktepe', 'kalaba'],
    'Etimesgut': ['etimesgut', 'elvankent', 'eryaman'],
    'Yenimahalle': ['yenimahalle', 'demetevler', 'batıkent'],
    'Mamak': ['mamak', 'natoyolu'],
    'Altındağ': ['altındağ', 'ulus', 'sıhhiye'],
    'Gölbaşı': ['gölbaşı', 'eymir'],
    'Pursaklar': ['pursaklar', 'saraycık'],
    'Polatlı': ['polatlı'],
    'Çubuk': ['çubuk'],
    'Bağlıca': ['bağlıca', 'incek']
}

def detect_region(route_text):
    """Detect region from route text"""
    if pd.isna(route_text):
        return 'Unknown'
    
    route_lower = route_text.lower()
    
    for region, keywords in regions.items():
        for keyword in keywords:
            if keyword in route_lower:
                return region
    
    return 'Center'  # Default

# Region labeling
df['REGION'] = df['GÜZERGAH'].apply(detect_region)

print("Region Distribution:")
print(df['REGION'].value_counts())

# Region-based summary
region_summary = df.groupby('REGION').agg({
    'HAT NO': 'nunique',
    'TAŞINAN YOLCU SAYISI': 'sum',
    'DOLULUK ORANI': 'mean',
    'SEFER SAYISI': 'sum'
}).rename(columns={
    'HAT NO': 'Route_Count',
    'TAŞINAN YOLCU SAYISI': 'Total_Passengers',
    'DOLULUK ORANI': 'Avg_Occupancy',
    'SEFER SAYISI': 'Total_Trips'
})

region_summary['Passengers_Per_Route'] = region_summary['Total_Passengers'] / region_summary['Route_Count']
region_summary = region_summary.sort_values('Total_Passengers', ascending=False)

print("\n" + "="*80)
print("REGIONAL PERFORMANCE SUMMARY")
print("="*80)
print(region_summary.round(1))

In [ ]:
# 3.2 - Regional Performance Visualization

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Total passengers by region
top_regions = region_summary.head(10)
axes[0, 0].barh(top_regions.index, top_regions['Total_Passengers'], color='steelblue', alpha=0.7)
axes[0, 0].set_title('Total Passengers by Region', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Total Passengers')
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Average occupancy by region
axes[0, 1].bar(range(len(top_regions)), top_regions['Avg_Occupancy'], color='coral', alpha=0.7)
axes[0, 1].set_xticks(range(len(top_regions)))
axes[0, 1].set_xticklabels(top_regions.index, rotation=45, ha='right')
axes[0, 1].axhline(df['DOLULUK ORANI'].mean(), color='green', linestyle='--', label='System Avg.')
axes[0, 1].set_title('Average Occupancy by Region', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Average Occupancy (%)')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Route count distribution
axes[1, 0].pie(top_regions['Route_Count'], labels=top_regions.index, autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Route Count Distribution by Region', fontsize=14, fontweight='bold')

# 4. Efficiency (passengers per route)
axes[1, 1].scatter(top_regions['Route_Count'], top_regions['Passengers_Per_Route'], 
                   s=top_regions['Total_Passengers']/1000, alpha=0.6, edgecolors='black')
for idx, region in enumerate(top_regions.index):
    axes[1, 1].annotate(region, 
                        (top_regions.iloc[idx]['Route_Count'], 
                         top_regions.iloc[idx]['Passengers_Per_Route']),
                        fontsize=8, alpha=0.7)
axes[1, 1].set_title('Region Efficiency (Route Count vs Passengers/Route)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Route Count')
axes[1, 1].set_ylabel('Passengers / Route')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# 4. Route Text Mining and Hub Detection

**Gemini Recommendation:**
> "Which stop names appear most frequently? (Kızılay, Ulus, Sıhhiye, Metro Stations)"

**Goal:**
- Detect transfer centers
- Find most important nodes
- Map the route network

In [ ]:
# 4.1 - Stop/Location Frequency Analysis

# Parse route texts
all_locations = []
for route_text in df['GÜZERGAH'].dropna():
    # Locations separated by dashes
    if '-' in route_text:
        locations = route_text.split('-')
    else:
        locations = [route_text]
    
    # Clean and add each location
    for loc in locations:
        loc_clean = loc.strip().lower()
        if len(loc_clean) > 2:  # Skip very short names
            all_locations.append(loc_clean)

# Frequency analysis
location_freq = pd.Series(all_locations).value_counts()

print("="*80)
print("MOST FREQUENT LOCATIONS (HUB CANDIDATES)")
print("="*80)
print(location_freq.head(30))

# Metro station detection
metro_keywords = ['metro', 'istasyon', 'garı']
metro_locations = location_freq[location_freq.index.str.contains('|'.join(metro_keywords))]

print("\n" + "="*80)
print("METRO STATIONS")
print("="*80)
print(metro_locations.head(20))

In [ ]:
# 4.2 - Hub (Transfer Center) Visualization

fig, axes = plt.subplots(2, 1, figsize=(18, 12))

# 1. Top 30 locations
top_30_loc = location_freq.head(30)
axes[0].barh(range(len(top_30_loc)), top_30_loc.values, color='steelblue', alpha=0.7)
axes[0].set_yticks(range(len(top_30_loc)))
axes[0].set_yticklabels(top_30_loc.index)
axes[0].set_title('Top 30 Most Frequent Locations (Hub Candidates)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Frequency')
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

# 2. Metro stations
if len(metro_locations) > 0:
    top_metro = metro_locations.head(15)
    axes[1].barh(range(len(top_metro)), top_metro.values, color='coral', alpha=0.7)
    axes[1].set_yticks(range(len(top_metro)))
    axes[1].set_yticklabels(top_metro.index)
    axes[1].set_title('Metro Stations (By Frequency)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Frequency')
    axes[1].grid(axis='x', alpha=0.3)
    axes[1].invert_yaxis()
else:
    axes[1].text(0.5, 0.5, 'No metro station data found', 
                 ha='center', va='center', fontsize=12)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\nHUB ANALYSIS:")
print(f"- Total {len(location_freq)} unique locations detected")
print(f"- Most popular hub: '{location_freq.index[0]}' ({location_freq.iloc[0]} occurrences)")
print(f"- Number of metro stations: {len(metro_locations)}")

---
# 5. Metro Integration Analysis

**Gemini Recommendation:**
> "Performance of bus routes feeding metro stations and their relationship with metro occupancy rates"

**Analysis:**
- Identify metro feeder routes
- Compare their performance
- Measure integration quality

In [ ]:
# 5.1 - Metro Feeder Route Detection

# Filter routes related to metro
df['Metro_Feeder'] = df['GÜZERGAH'].str.lower().str.contains(
    'metro|istasyon', 
    case=False, 
    na=False
)

# Metro feeder vs normal routes
metro_feeder_df = df[df['Metro_Feeder'] == True]
normal_routes_df = df[df['Metro_Feeder'] == False]

# Comparative statistics
comparison = pd.DataFrame({
    'Metro Feeder Routes': [
        metro_feeder_df['HAT NO'].nunique(),
        metro_feeder_df['TAŞINAN YOLCU SAYISI'].sum(),
        metro_feeder_df['DOLULUK ORANI'].mean(),
        metro_feeder_df.groupby('HAT NO')['TAŞINAN YOLCU SAYISI'].sum().mean()
    ],
    'Normal Routes': [
        normal_routes_df['HAT NO'].nunique(),
        normal_routes_df['TAŞINAN YOLCU SAYISI'].sum(),
        normal_routes_df['DOLULUK ORANI'].mean(),
        normal_routes_df.groupby('HAT NO')['TAŞINAN YOLCU SAYISI'].sum().mean()
    ]
}, index=['Route Count', 'Total Passengers', 'Avg. Occupancy (%)', 'Passengers per Route'])

print("="*80)
print("METRO FEEDER ROUTES vs NORMAL ROUTES")
print("="*80)
print(comparison.round(1))

# Busiest metro feeder routes
metro_route_summary = metro_feeder_df.groupby('HAT NO').agg({
    'TAŞINAN YOLCU SAYISI': 'sum',
    'DOLULUK ORANI': 'mean',
    'GÜZERGAH': 'first'
}).sort_values('TAŞINAN YOLCU SAYISI', ascending=False)

print("\n" + "="*80)
print("BUSIEST METRO FEEDER ROUTES")
print("="*80)
print(metro_route_summary.head(10))

In [ ]:
# 5.2 - Metro Integration Visualization

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Route count comparison
route_comparison = [metro_feeder_df['HAT NO'].nunique(), 
                     normal_routes_df['HAT NO'].nunique()]
axes[0, 0].bar(['Metro Feeder', 'Normal'], route_comparison, 
               color=['#3498db', '#95a5a6'], alpha=0.7)
axes[0, 0].set_title('Route Count Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Route Count')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Average occupancy comparison
occupancy_comparison = [metro_feeder_df['DOLULUK ORANI'].mean(),
                         normal_routes_df['DOLULUK ORANI'].mean()]
axes[0, 1].bar(['Metro Feeder', 'Normal'], occupancy_comparison,
               color=['#e74c3c', '#95a5a6'], alpha=0.7)
axes[0, 1].set_title('Average Occupancy Comparison', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Occupancy (%)')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Top 10 metro feeder routes
top_10_metro = metro_route_summary.head(10)
axes[1, 0].barh(top_10_metro.index.astype(str), 
                top_10_metro['TAŞINAN YOLCU SAYISI'],
                color='steelblue', alpha=0.7)
axes[1, 0].set_title('Top 10 Busiest Metro Feeder Routes', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Total Passengers')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Occupancy distribution (box plot)
data_to_plot = [metro_feeder_df['DOLULUK ORANI'].dropna(),
                normal_routes_df['DOLULUK ORANI'].dropna()]
axes[1, 1].boxplot(data_to_plot, labels=['Metro Feeder', 'Normal'])
axes[1, 1].set_title('Occupancy Distribution (Box Plot)', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Occupancy (%)')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
# 6. Anomaly Detection

**Gemini Recommendation:**
> "System can alert when a route's passenger count falls outside standard deviation"

**Methodology:**
- Z-score based anomaly detection
- Outlier values in daily passenger count for each route
- Possible causes: Breakdown, event, weather

In [ ]:
# 6.1 - Z-Score Based Anomaly Detection

# Anomaly analysis for top 10 busiest routes
top_10_routes = df.groupby('HAT NO')['TAŞINAN YOLCU SAYISI'].sum().nlargest(10).index

anomaly_report = []

for route in top_10_routes:
    route_df = df[df['HAT NO'] == route].copy()
    route_df = route_df.sort_values('TARIH')
    
    # Calculate Z-score
    z_scores = np.abs(stats.zscore(route_df['TAŞINAN YOLCU SAYISI']))
    
    # Threshold: 3 (classical anomaly threshold)
    anomaly_mask = z_scores > 3
    anomalies = route_df[anomaly_mask]
    
    if len(anomalies) > 0:
        for _, row in anomalies.iterrows():
            anomaly_report.append({
                'Route No': route,
                'Date': row['TARIH'].date(),
                'Passengers': row['TAŞINAN YOLCU SAYISI'],
                'Average': route_df['TAŞINAN YOLCU SAYISI'].mean(),
                'Std': route_df['TAŞINAN YOLCU SAYISI'].std(),
                'Z-Score': z_scores[row.name],
                'Deviation': row['TAŞINAN YOLCU SAYISI'] - route_df['TAŞINAN YOLCU SAYISI'].mean()
            })

anomaly_df = pd.DataFrame(anomaly_report)

print("="*100)
print("ANOMALY DETECTION REPORT (Z-Score > 3)")
print("="*100)
if len(anomaly_df) > 0:
    print(f"Total {len(anomaly_df)} anomalies detected.\n")
    print(anomaly_df.round(1).to_string(index=False))
else:
    print("No anomalies detected.")

print("\nNOTE: Z-Score > 3 = Statistically outlier value")
print("Positive deviation = More passengers than expected (event?)")
print("Negative deviation = Fewer passengers than expected (breakdown, road work?)")

In [ ]:
# 6.2 - Anomaly Visualization

# Anomaly graph for first 3 routes
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for idx, route in enumerate(top_10_routes[:3]):
    route_df = df[df['HAT NO'] == route].sort_values('TARIH')
    
    # Statistics
    mean_val = route_df['TAŞINAN YOLCU SAYISI'].mean()
    std_val = route_df['TAŞINAN YOLCU SAYISI'].std()
    
    # Anomaly detection
    z_scores = np.abs(stats.zscore(route_df['TAŞINAN YOLCU SAYISI']))
    anomaly_mask = z_scores > 3
    
    # Plot
    axes[idx].plot(route_df['TARIH'], route_df['TAŞINAN YOLCU SAYISI'],
                   linewidth=2, color='steelblue', label='Daily Passengers')
    axes[idx].axhline(mean_val, color='green', linestyle='--', label=f'Average: {mean_val:.0f}')
    axes[idx].axhline(mean_val + 3*std_val, color='red', linestyle=':', 
                      label=f'Upper Threshold (+3σ)', alpha=0.7)
    axes[idx].axhline(mean_val - 3*std_val, color='red', linestyle=':', 
                      label=f'Lower Threshold (-3σ)', alpha=0.7)
    
    # Mark anomalies
    if anomaly_mask.any():
        axes[idx].scatter(route_df[anomaly_mask]['TARIH'],
                         route_df[anomaly_mask]['TAŞINAN YOLCU SAYISI'],
                         color='red', s=100, zorder=5, label='Anomaly')
    
    axes[idx].set_title(f'Route {route} - Anomaly Detection', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Passenger Count')
    axes[idx].legend(loc='upper right')
    axes[idx].grid(True, alpha=0.3)

axes[2].set_xlabel('Date')
plt.tight_layout()
plt.show()

---
# 7. Summary Report and Action Recommendations

**Executive Summary Based on Gemini Dashboard Recommendations**

In [ ]:
# 7.1 - Executive Summary Report

print("="*100)
print("EGO SYSTEM - EXECUTIVE SUMMARY REPORT")
print("="*100)
print(f"Report Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Data Period: {df['TARIH'].min().date()} → {df['TARIH'].max().date()}")
print(f"Days Analyzed: {df['TARIH'].nunique()}")
print(f"Total Routes: {df['HAT NO'].nunique()}")

print("\n" + "="*100)
print("1. SYSTEM HEALTH INDICATORS")
print("="*100)
print(f"Daily Average Passengers: {df.groupby('TARIH')['TAŞINAN YOLCU SAYISI'].sum().mean():,.0f}")
print(f"System Average Occupancy: {df['DOLULUK ORANI'].mean():.1f}%")
print(f"Daily Average Trips: {df.groupby('TARIH')['SEFER SAYISI'].sum().mean():,.0f}")

print("\n" + "="*100)
print("2. ALERT STATUS")
print("="*100)
print(route_metrics['Alert'].value_counts())

if len(critical) > 0:
    print(f"\n🔴 CRITICAL ALERT: {len(critical)} routes operating at 90%+ occupancy!")
    print("   Urgent capacity increase recommended: ", list(critical.head(3).index))

if len(ghost) > 0:
    print(f"\n🟢 SAVINGS OPPORTUNITY: {len(ghost)} routes operating at 10%- occupancy!")
    print("   Trip optimization recommended: ", list(ghost.head(3).index))

print("\n" + "="*100)
print("3. REGIONAL PERFORMANCE")
print("="*100)
print("Top 5 Busiest Regions:")
print(region_summary[['Total_Passengers', 'Avg_Occupancy']].head(5).round(1))

print("\n" + "="*100)
print("4. METRO INTEGRATION")
print("="*100)
print(f"Metro Feeder Routes: {metro_feeder_df['HAT NO'].nunique()}")
print(f"Metro Feeder Average Occupancy: {metro_feeder_df['DOLULUK ORANI'].mean():.1f}%")
print(f"Normal Routes Average Occupancy: {normal_routes_df['DOLULUK ORANI'].mean():.1f}%")

if len(anomaly_df) > 0:
    print("\n" + "="*100)
    print("5. ANOMALY DETECTIONS")
    print("="*100)
    print(f"Total {len(anomaly_df)} anomalies detected.")
    print("Last 5 anomalies:")
    print(anomaly_df.tail(5)[['Route No', 'Date', 'Passengers', 'Deviation']].round(0).to_string(index=False))

print("\n" + "="*100)
print("6. RECOMMENDATIONS")
print("="*100)
print("✓ Allocate additional vehicles to critical routes")
print("✓ Re-evaluate schedules for low-occupancy routes")
print("✓ Optimize metro feeder route schedules")
print("✓ Detailed investigation of days with detected anomalies")
print("✓ Regional-based capacity planning")

print("\n" + "="*100)

---
# ANALYSIS COMPLETED ✓

## Implemented Gemini Recommendations:

1. ✅ **Occupancy Heatmap** - Route × Date and weekly patterns
2. ✅ **Operational Efficiency** - Alert system, ghost routes, critical routes
3. ✅ **Regional Performance** - District-based analysis and groupings
4. ✅ **Route Text Mining** - Hub detection, most popular locations
5. ✅ **Metro Integration** - Feeder route analysis and comparison
6. ✅ **Anomaly Detection** - Z-score based statistical detection
7. ✅ **Executive Report** - Summary and action recommendations

---

**Usage:**
- All analyses can be run with Run All
- Each section can run independently
- Results are presented with English explanations

**Data Source:** `ego_data_clean.csv`

**Reference:** `GEMINI_ANALIZ_ONERILERI.md`